# 21 — Longer-history XGBoost: history and search budget

**Optional after 08 and a completed 09b history study.** Use the verified project kernel and **Restart Kernel and Run All**. No other research notebook needs to run first. Existing 2024 caches and preparation are required; no downloads or automatic preparation run here.

Use **the same 2024–2025 training populations, 22 features and three expanding folds as longer-history LightGBM**. First refit the original 16 parameter configurations on that longer history: their old 2025 scores cannot be reused. Then complete all 54 XGBoost configurations. Exact duplicate settings reuse scores within this new study only.

Compare original 2025 selected settings → those same settings with added history → first-16 longer-history selection → completed larger search. This separates history from retuning. Existing hyperparameter ranges and pipeline stay fixed. Equal data does not make different search spaces equally exhaustive.

LR has a continuous C range, so 100/200 proposals can test new settings. RF/XGBoost have only 48/54 finite combinations: once all are tested, 100 or 200 recorded duplicates add no information. Their requested cap is therefore bounded by the remaining grid. Report distinct configurations and completed trials honestly.

Each successful trial is checkpointed. Restart/Run All resumes compatible progress; completed result folders remain archived. Choose settings from mean fold AP, **then** read January–March validation once and report all candidates. This period is already exposed, so results remain exploratory. No test/November outcomes, source corrections, classifier replacement or dashboard deployment occurs. A larger search does not guarantee 0.70 AP.

## 1. Verify the original classifier and matching longer-history preparation

The classifier supplies its search space/pipeline and original control. 09b supplies only verified historical-data provenance and expected row support. Original scores are not reused for the new 2024–2025 fits. Budget is the only user setting below.

In [ ]:
import sys
from pathlib import Path
REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open this notebook inside the FYP repository.")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
import ast
import hashlib
import itertools
import json
from datetime import datetime, timezone
from importlib import metadata, reload
from tempfile import TemporaryDirectory
from time import perf_counter
import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_score,
                             recall_score, f1_score, precision_recall_curve)
import xgboost as xgb
from xgboost import XGBClassifier
import pricecatcher_config as cfg
cfg = reload(cfg)

FAMILY = 'XGBoost'
PARENT_STEM = '08_xgboost'
MODEL_FILE = 'xgboost.joblib'
FINITE_GRID = True
TARGET_TRIALS = 200  # LR: 100 or 200 total; finite grids stop once complete.
if TARGET_TRIALS not in [100, 200]:
    raise ValueError("Use the declared 100/200 requested cap.")
def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

parents = sorted((cfg.PROJECT_ROOT / PARENT_STEM).glob("*/study_manifest.json"), reverse=True)
history_runs = sorted((cfg.PROJECT_ROOT / "09b_longer_history_lightgbm").glob("*/optuna/study_manifest.json"), reverse=True)
if not parents or not history_runs:
    raise FileNotFoundError("Complete the original classifier and matching 09b direct-history study first.")
parent_dir, history_dir = parents[0].parent, history_runs[0].parent
parent, history_parent = cfg.verify_saved_study(parent_dir), cfg.verify_saved_study(history_dir)
selection = parent["selection"]
if selection["family"] != FAMILY or selection["trials"] != 16:
    raise ValueError("Expected the matching original 16-trial classifier.")
if parent["input_sha256"] != history_parent["input_sha256"]:
    raise ValueError("Classifier and historical preparation use different input snapshots.")
# Legacy classifier manifests omitted Python; their saved runs used Python 3.12.
parent_python = parent.get("python_version", "3.12")
if "python_version" not in parent:
    print("Legacy parent manifest: using the documented Python 3.12 requirement.")
if sys.version_info[:2] != tuple(map(int, parent_python.split(".")[:2])):
    raise ValueError(f"Use the parent classifier's Python {parent_python} environment.")
packages = {p: metadata.version(p) for p in ["optuna", "numpy", "pandas", "scikit-learn", "joblib"]}
if any(value != parent["package_versions"][p] for p, value in packages.items()):
    raise ValueError("Use the parent classifier's saved package versions.")
if FAMILY == "XGBoost" and xgb.__version__ != parent["package_versions"]["xgboost_module"]:
    raise ValueError("Use the saved XGBoost module version (distribution name may differ across platforms).")
if FAMILY == "XGBoost":
    packages["xgboost_module"] = xgb.__version__
FOLDS = ["fold_1", "fold_2", "fold_3"]
folders = {name: cfg.PROJECT_ROOT / "experiments" / name / "05_modelling" for name in FOLDS + ["holdout"]}
settings_by_fold = {name: json.loads((folder / "experiment_settings.json").read_text()) for name, folder in folders.items()}
settings = settings_by_fold["holdout"]
if any(settings["preparation"][key] != value for key, value in cfg.PREPARATION_SETTINGS.items()):
    raise ValueError("Current configuration differs from saved preparation.")
FEATURES, NUMERIC, CATEGORICAL = (settings[k] for k in ["feature_columns", "numeric_features", "categorical_features"])
if FEATURES != selection["feature_columns"] or FEATURES != history_parent["selection"]["feature_columns"]:
    raise ValueError("Feature lists differ.")
SEED, N_JOBS = selection["seed"], selection.get("n_jobs", 2)
TREES = selection.get("fixed_parameters", {}).get("n_estimators", 150)
MAX_ITER = selection.get("max_iter", 2000)
SEARCH_SPACE = selection["search"]
parent_manifest_sha256 = file_hash(parent_dir / "study_manifest.json")
history_manifest_sha256 = file_hash(history_dir / "study_manifest.json")
configuration_sha256 = file_hash(Path(cfg.__file__))
print("Family:", FAMILY, "| history: 2024–2025 | requested cap:", TARGET_TRIALS)
print("Original parent:", parent_dir)
print("Shared historical preparation:", history_dir)
print("Search space:", SEARCH_SPACE)

## 2. Reuse reviewed training and model functions

Load only the named functions from 09b and the original model notebook, not executable notebook cells. Their transformations remain training-fitted. Rebuild original features/labels before adding history, and verify counts against 09b. LR convergence warnings stop the study; do not quietly compare unconverged fits.

In [ ]:
history_notebook = REPOSITORY_ROOT / "09b_longer_history_lightgbm.ipynb"
source_nb = json.loads(history_notebook.read_text())
functions = []
for cell in source_nb["cells"]:
    if cell["cell_type"] == "code":
        tree = ast.parse("".join(cell["source"]))
        functions.extend(node for node in tree.body if isinstance(node, ast.FunctionDef)
                         and node.name in {"training_table", "prepare_training"})
if sorted(node.name for node in functions) != ["prepare_training", "training_table"]:
    raise ValueError("09b training helpers changed; review compatibility before this study.")
calculation = ast.Module(body=functions, type_ignores=[])
helper_sha256 = hashlib.sha256(ast.dump(calculation).encode()).hexdigest()
exec(compile(calculation, str(history_notebook), "exec"), globals())
helper_notebook_sha256 = file_hash(history_notebook)

additional_inputs = history_parent["additional_input_sha256"]
historical_paths = [cfg.resolve_input(path) for path in additional_inputs
                    if cfg.resolve_input(path).name == "selected_prices_2024.parquet"]
if len(historical_paths) != 1:
    raise ValueError("Expected the exact 2024 source snapshot recorded by 09b.")
earlier_source = pd.read_parquet(historical_paths[0], columns=cfg.GROUP_KEYS + ["date", "premise_code", "price"])
earlier_source["date"] = pd.to_datetime(earlier_source.date)
if earlier_source.empty or not earlier_source.date.between("2024-01-01", "2024-12-31").all():
    raise ValueError("Supplementary source must contain only the recorded 2024 observations.")
features, keys = FEATURES, cfg.GROUP_KEYS + ["week_start"]
training_rows, cleaning_rows = [], []
expected_support = pd.read_csv(history_dir / "training_comparison.csv")

def matching_training(name):
    training, evaluation = prepare_training(name)
    for history, recorded in [("2025-only", training_rows[-2]), ("2024-added", training_rows[-1])]:
        expected = expected_support.loc[expected_support.fold.eq(name) & expected_support.history.eq(history)]
        if len(expected) != 1:
            raise ValueError(f"{name}: missing original support evidence.")
        for column in ["rows", "positives", "first_week", "last_outcome"]:
            if str(recorded[column]) != str(expected.iloc[0][column]):
                raise ValueError(f"{name}: rebuilt {history} {column} differs from 09b.")
    if training.surge_label.nunique() != 2 or evaluation.surge_label.nunique() != 2:
        raise ValueError("Both classes are needed for the overall AP comparison.")
    return training, evaluation

In [ ]:
model_notebook = REPOSITORY_ROOT / (PARENT_STEM + ".ipynb")
model_functions = []
for cell in json.loads(model_notebook.read_text())["cells"]:
    if cell["cell_type"] == "code":
        model_functions.extend(node for node in ast.parse("".join(cell["source"])).body
            if isinstance(node, ast.FunctionDef) and node.name in {"model_inputs", "build_pipeline", "fit_and_score"})
if sorted(node.name for node in model_functions) != ["build_pipeline", "fit_and_score", "model_inputs"]:
    raise ValueError("Parent pipeline helpers changed; review compatibility.")
model_tree = ast.Module(body=model_functions, type_ignores=[])
model_helper_sha256 = hashlib.sha256(ast.dump(model_tree).encode()).hexdigest()
exec(compile(model_tree, str(model_notebook), "exec"), globals())

def fit_candidate(parameters, training, evaluation):
    if FAMILY == "Logistic Regression":
        return fit_and_score(parameters["C"], parameters["class_weight"], training, evaluation)
    return fit_and_score(parameters, training, evaluation)

def evaluate(y, scores, cutoff):
    y = np.asarray(y, dtype=int)
    alerts = np.asarray(scores) >= cutoff
    tn, fp, fn, tp = confusion_matrix(y, alerts, labels=[0, 1]).ravel()
    return {"rows": len(y), "positives": int(y.sum()), "average_precision":
        float(average_precision_score(y, scores)) if len(np.unique(y)) == 2 else np.nan,
        "cutoff": cutoff, "precision": precision_score(y, alerts, zero_division=0),
        "recall": recall_score(y, alerts, zero_division=0) if y.sum() else np.nan,
        "f1": f1_score(y, alerts, zero_division=0) if y.sum() else np.nan,
        "alerts": int(alerts.sum()), "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

## 3. Declare configurations and restore compatible progress

Original parameter configurations are queued in their recorded order, but every score is re-evaluated on longer history. For finite spaces, enumerate all remaining configurations once in seeded order. The checkpoint contains only completed trials and their new fold metrics. Duplicate original proposals remain recorded but use no extra fits.

LR can extend 100 → 200 by changing only the cap. For RF/XGBoost, a cap above the completed finite space does not create more unique settings. Completing a grid identifies the highest fold-AP setting **within this declared grid**, not a globally optimal model or optimum future-performance budget.

In [ ]:
this_notebook = REPOSITORY_ROOT / "research" / "21_longer_history_xgboost.ipynb"
def calculation_hash():
    nb = json.loads(this_notebook.read_text())
    tree = ast.parse("\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"))
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "TARGET_TRIALS" for t in node.targets):
            node.value = ast.Constant(value=0)
    return hashlib.sha256(ast.dump(tree).encode()).hexdigest()

source_trials = pd.read_csv(parent_dir / "optuna_trials.csv", keep_default_na=False, float_precision="round_trip").sort_values("number")
if source_trials.number.tolist() != list(range(16)) or not source_trials.state.eq("COMPLETE").all():
    raise ValueError("Expected all original 16 completed parameter configurations.")
def parent_parameters(row):
    parameters = {}
    for name, specification in SEARCH_SPACE.items():
        raw = row["params_" + name]
        if isinstance(specification, dict):
            parameters[name] = float(raw)
        else:
            choices = [c for c in specification if (c is None and str(raw).lower() in ["", "none"])
                       or (c is not None and str(c) == str(raw))]
            if len(choices) != 1:
                raise ValueError(f"Unknown recorded choice: {name}={raw}")
            parameters[name] = choices[0]
    return parameters
initial_parameters = [parent_parameters(row) for _, row in source_trials.iterrows()]
if initial_parameters[selection["selected_trial"]] != selection["parameters"]:
    raise ValueError("Parent's selected trial and recorded parameters disagree.")
ordered_parameters = list(initial_parameters)
if FINITE_GRID:
    names = list(SEARCH_SPACE)
    grid = [dict(zip(names, values)) for values in itertools.product(*(SEARCH_SPACE[n] for n in names))]
    initial_keys = {json.dumps(p, sort_keys=True) for p in initial_parameters}
    missing = [p for p in grid if json.dumps(p, sort_keys=True) not in initial_keys]
    np.random.default_rng(SEED).shuffle(missing)
    ordered_parameters += missing
    total_trials = min(TARGET_TRIALS, len(ordered_parameters))
    print("Finite configurations:", len(grid), "| total recorded trials including original duplicates:", total_trials)
else:
    total_trials = TARGET_TRIALS
distributions = {name: (optuna.distributions.FloatDistribution(**spec) if isinstance(spec, dict)
                         else optuna.distributions.CategoricalDistribution(spec)) for name, spec in SEARCH_SPACE.items()}
identity = {"family": FAMILY, "parent_sha256": parent_manifest_sha256, "history_sha256": history_manifest_sha256,
    "configuration_sha256": configuration_sha256, "history_helper_sha256": helper_sha256,
    "model_helper_sha256": model_helper_sha256, "code_sha256": calculation_hash(), "seed": SEED,
    "search_space": SEARCH_SPACE, "history_start": "2024-01", "feature_columns": FEATURES,
    "python_version": sys.version.split()[0], "package_versions": packages,
    "policy": "refit parent configurations on shared history; finite grid completion or continuous seeded TPE"}
study_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
work = cfg.PROJECT_ROOT / "research" / "21_longer_history_xgboost" / study_id
work.mkdir(parents=True, exist_ok=True)
checkpoint = work / "checkpoint.joblib"
if checkpoint.exists():
    saved = joblib.load(checkpoint)
    if saved["identity"] != identity:
        raise ValueError("Incompatible checkpoint; preserve it and investigate.")
    study, trial_rows = saved["study"], saved["trial_rows"]
else:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.NopPruner())
    trial_rows = []
trial_cache = {}
recorded = pd.DataFrame(trial_rows)
if len(trial_rows) != 3 * len(study.trials) or len(study.trials) > total_trials:
    raise ValueError("Checkpoint trial count differs from completed fold rows or requested cap.")
for trial in study.trials:
    rows = recorded.loc[recorded.trial.eq(trial.number)]
    if trial.state != optuna.trial.TrialState.COMPLETE or len(rows) != 3 or set(rows.fold) != set(FOLDS):
        raise ValueError("Each checkpointed trial needs exactly three completed folds.")
    if any(json.loads(p) != trial.params for p in rows.parameters_json):
        raise ValueError("Checkpoint parameter records differ.")
    np.testing.assert_allclose(rows.average_precision.mean(), trial.value, rtol=1e-10, atol=1e-12)
    if trial.number < len(ordered_parameters) and (FINITE_GRID or trial.number < 16):
        if trial.params != ordered_parameters[trial.number]:
            raise ValueError("Checkpoint does not follow the fixed parameter sequence.")
    trial_cache.setdefault(json.dumps(trial.params, sort_keys=True), rows.to_dict(orient="records"))
print("Completed:", len(study.trials), "| remaining:", total_trials - len(study.trials), "| checkpoint:", checkpoint)

## 4. Run three training-only fits per new configuration

Use temporary training files, one fold at a time in memory. Repeated settings reuse their new-history results, never old 2025 metrics. All completed trials checkpoint atomically. January–March validation is not opened until the whole requested/capped search finishes. If interrupted, Restart Kernel and Run All resumes the last completed trial; an incomplete trial may need repeating.

In [ ]:
def save_checkpoint():
    temporary = checkpoint.with_suffix(".tmp")
    joblib.dump({"identity": identity, "study": study, "trial_rows": trial_rows}, temporary)
    temporary.replace(checkpoint)

def objective(trial):
    parameters = {name: (trial.suggest_float(name, **spec) if isinstance(spec, dict)
                         else trial.suggest_categorical(name, spec)) for name, spec in SEARCH_SPACE.items()}
    key = json.dumps(parameters, sort_keys=True)
    if key in trial_cache:
        rows = [{**r, "trial": trial.number, "reused_from_trial": r["trial"], "fit_and_score_seconds": 0.0}
                for r in trial_cache[key]]
    else:
        rows = []
        for name in FOLDS:
            training = pd.read_parquet(training_paths[name])
            evaluation = fold_validation[name]
            started = perf_counter()
            model, scores = fit_candidate(parameters, training, evaluation)
            rows.append({"trial": trial.number, "fold": name, "parameters_json": key,
                "fit_and_score_seconds": perf_counter() - started, **evaluate(evaluation.surge_label, scores, 0.5)})
            del training, model
        trial_cache[key] = rows
    trial_rows.extend(rows)
    return float(np.mean([r["average_precision"] for r in rows]))

parent_folds = pd.read_csv(parent_dir / "fold_comparison.csv")
parent_folds = parent_folds.loc[parent_folds.method.eq("Optuna selected")]
if len(parent_folds) != 3 or set(parent_folds.fold) != set(FOLDS):
    raise ValueError("Parent needs three selected-configuration fold results.")
save_checkpoint()
if len(study.trials) < total_trials:
    with TemporaryDirectory(prefix="hargalens_history_search_") as temporary:
        training_paths, fold_validation = {}, {}
        for name in FOLDS:
            training, evaluation = matching_training(name)
            old = parent_folds.set_index("fold").loc[name]
            if int(old.rows) != len(evaluation) or int(old.positives) != int(evaluation.surge_label.sum()):
                raise ValueError("Original and longer-history evaluation rows differ.")
            training_paths[name] = Path(temporary) / (name + ".parquet")
            training[FEATURES + ["surge_label"]].to_parquet(training_paths[name], index=False)
            fold_validation[name] = evaluation
            del training
        while len(study.trials) < total_trials:
            number = len(study.trials)
            if FINITE_GRID or number < 16:
                study.enqueue_trial(ordered_parameters[number])
            trial = study.ask()
            value = objective(trial)
            study.tell(trial, value)
            save_checkpoint()
            print(f"Completed {len(study.trials)}/{total_trials} | distinct {len(trial_cache)} | best AP {study.best_value:.4f}", flush=True)
        del fold_validation
trial_folds = pd.DataFrame(trial_rows)
if FINITE_GRID and len(trial_cache) != len(grid):
    raise ValueError("Finite grid was not fully covered.")
fixed_trial = study.trials[selection["selected_trial"]]
winners = {"Added history, original selected settings": fixed_trial,
           "Added history, best first 16 proposals": max(study.trials[:16], key=lambda t: (t.value, -t.number))}
for budget in ([total_trials] if FINITE_GRID else [n for n in [100, 200] if n <= total_trials]):
    label = "Added history, complete grid" if FINITE_GRID else f"Added history, best first {budget} proposals"
    winners[label] = max(study.trials[:budget], key=lambda t: (t.value, -t.number))
final_trial = max(study.trials, key=lambda t: (t.value, -t.number))
fold_summary = pd.DataFrame([{"method": "Original 2025 selected", "selected_trial": selection["selected_trial"],
    "mean_fold_ap": parent_folds.average_precision.mean(), "std_fold_ap": parent_folds.average_precision.std()}] +
    [{"method": label, "selected_trial": t.number, "mean_fold_ap": t.value,
      "std_fold_ap": trial_folds.loc[trial_folds.trial.eq(t.number), "average_precision"].std()}
     for label, t in winners.items()])
display(fold_summary)
print("History/retuning comparisons fixed before later validation. No validation-based reselection.")

## 5. Evaluate later validation once and save all comparisons

Verify the original model reproduces its saved validation scores, then fit longer-history candidates through December 2025. Duplicate winners reuse one final fit. Select each new score cutoff using January–March F1, highest cutoff for ties. These are apparent cutoff-selection metrics, not independent tests. Report both a common 0.5 reference and each selected cutoff. The final research candidate remains the completed-search fold winner.

In [ ]:
training, validation = matching_training("holdout")
prediction_keys = ["item_code", "state", "district", "premise_type", "project_group", "week_start", "next_week_end", "surge_label"]
ids = prediction_keys[:-1]
validation = validation.sort_values(ids).reset_index(drop=True)
saved_predictions = pd.read_parquet(parent_dir / "validation_predictions.parquet").sort_values(ids).reset_index(drop=True)
pd.testing.assert_frame_equal(validation[prediction_keys], saved_predictions[prediction_keys])
if validation.duplicated(ids).any():
    raise ValueError("Duplicate validation identifiers.")
original_model = joblib.load(parent_dir / MODEL_FILE)
original_scores = original_model.predict_proba(model_inputs(validation))[:, 1]
np.testing.assert_allclose(original_scores, saved_predictions.model_score, rtol=1e-9, atol=1e-11)
score_sets = {"Original 2025 selected": (original_scores, float(selection["score_cutoff"]))}
fitted = {}
for label, trial in winners.items():
    key = json.dumps(trial.params, sort_keys=True)
    if key not in fitted:
        model, scores = fit_candidate(trial.params, training, validation)
        p, r, cutoffs = precision_recall_curve(validation.surge_label, scores)
        f1 = np.divide(2*p[:-1]*r[:-1], p[:-1]+r[:-1], out=np.zeros(len(cutoffs)), where=(p[:-1]+r[:-1]) > 0)
        cutoff = float(cutoffs[np.flatnonzero(np.isclose(f1, f1.max(), rtol=0, atol=1e-12))[-1]])
        fitted[key] = (model, scores, cutoff)
    _, scores, cutoff = fitted[key]
    score_sets[label] = (scores, cutoff)
validation_rows, group_rows, comparison_rows = [], [], []
predictions = validation[prediction_keys].copy()
for number, (label, (scores, cutoff)) in enumerate(score_sets.items()):
    fold = fold_summary.loc[fold_summary.method.eq(label)].iloc[0]
    comparison_rows.append({"method": label, "mean_fold_ap": fold.mean_fold_ap,
        **evaluate(validation.surge_label, scores, cutoff)})
    for role, threshold in [("common 0.5 reference", 0.5), ("apparent cutoff-selection performance", cutoff)]:
        validation_rows.append({"method": label, "role": role, **evaluate(validation.surge_label, scores, threshold)})
    for group, positions in validation.groupby("project_group").indices.items():
        group_rows.append({"method": label, "project_group": group,
            **evaluate(validation.iloc[positions].surge_label, scores[positions], cutoff)})
    predictions[f"score_{number}"] = scores
    predictions[f"alert_{number}"] = scores >= cutoff
final_model, _, final_cutoff = fitted[json.dumps(final_trial.params, sort_keys=True)]
comparison = pd.DataFrame(comparison_rows)
validation_comparison, validation_groups = pd.DataFrame(validation_rows), pd.DataFrame(group_rows)
display(comparison)
display(validation_groups)
del training, original_model

## 6. Save separate research evidence

The resumable checkpoint remains separate from immutable completed results. The original model, all dashboard artifacts and Notebook 10 remain unchanged. The trained research model requires its matching original input pipeline and settings; it is not an automatically deployed replacement.

In [ ]:
cfg.verify_saved_study(parent_dir)
cfg.verify_saved_study(history_dir)
if (file_hash(parent_dir / "study_manifest.json") != parent_manifest_sha256
    or file_hash(history_dir / "study_manifest.json") != history_manifest_sha256
    or file_hash(Path(cfg.__file__)) != configuration_sha256 or calculation_hash() != identity["code_sha256"]):
    raise ValueError("Upstream study or calculation changed during research.")
output = work / "results" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
output.mkdir(parents=True, exist_ok=False)
trial_summary = pd.DataFrame([{"trial": t.number, "mean_fold_ap": t.value,
    "parameters_json": json.dumps(t.params, sort_keys=True),
    "new_fit_seconds": trial_folds.loc[trial_folds.trial.eq(t.number), "fit_and_score_seconds"].sum()}
    for t in study.trials])
trial_summary["best_fold_ap_so_far"] = trial_summary.mean_fold_ap.cummax()
for filename, table in {"budget_comparison.csv": comparison, "fold_summary.csv": fold_summary,
    "trial_summary.csv": trial_summary, "trial_fold_metrics.csv": trial_folds,
    "validation_comparison.csv": validation_comparison, "validation_group_metrics.csv": validation_groups,
    "training_support.csv": expected_support}.items():
    table.to_csv(output / filename, index=False)
predictions.to_parquet(output / "validation_predictions.parquet", index=False)
joblib.dump(final_model, output / "research_model.joblib")
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(trial_summary.trial+1, trial_summary.best_fold_ap_so_far)
ax.set(xlabel="Recorded new-history trials (duplicates may reuse results)", ylabel="Best mean fold AP",
       title=FAMILY + ": search progress is not independent accuracy", ylim=(0, 1))
fig.tight_layout(); fig.savefig(output / "search_progress.png", dpi=150); plt.show()
record = {"status": "completed", "family": FAMILY, "identity": identity,
    "requested_trial_cap": TARGET_TRIALS, "completed_trials": total_trials,
    "distinct_configurations": len(trial_cache), "finite_space_exhausted": FINITE_GRID,
    "selected_trial": final_trial.number, "parameters": final_trial.params,
    "fixed_parameters": selection.get("fixed_parameters", {"max_iter": MAX_ITER}),
    "seed": SEED, "n_jobs": N_JOBS if FAMILY != "Logistic Regression" else None, "feature_columns": FEATURES, "score_cutoff": final_cutoff,
    "training_start": "2024-01", "training_end": settings_by_fold["holdout"]["train_end"],
    "validation_end": settings_by_fold["holdout"]["validation_end"], "surge_threshold_pct": 10.0,
    "selection_rule": "highest mean fold AP in completed search; earliest trial breaks ties; no validation reselection",
    "score_interpretation": "uncalibrated model score", "validation_role": "already-inspected exploratory comparison and F1 cutoff selection",
    "history_support": cfg.input_reference(history_dir / "training_comparison.csv"),
    "parent_study": cfg.input_reference(parent_dir / "study_manifest.json"),
    "history_study": cfg.input_reference(history_dir / "study_manifest.json"),
    "input_sha256": parent["input_sha256"], "additional_input_sha256": history_parent["additional_input_sha256"],
    "dashboard_changed": False, "independent_test_evaluated": False,
    "prediction_columns": {str(i): label for i, label in enumerate(score_sets)},
    "output_sha256": {p.name: file_hash(p) for p in output.iterdir()}}
(output / "research_manifest.json").write_text(json.dumps(record, indent=2))
print("Saved:", output)
print("Completed trials:", total_trials, "| distinct configurations:", len(trial_cache))
print("Share budget_comparison.csv, fold_summary.csv, trial_summary.csv and validation_group_metrics.csv.")

## Report interpretation

Compare original 2025 selection with the **same selected settings plus 2024** to discuss history. Compare longer-history first-16 selection with the completed larger search to discuss extra tuning. Report all periods even if they decline. A selected maximum increasing with more trials is not an independent gain.

RF/XGBoost complete their declared finite grids; this is stronger within-space coverage than 200 repeated proposals, but not exhaustive over every possible estimator parameter. LR's 200 proposals do not establish a globally optimal C. Compare distinct settings and runtime, not only recorded trial counts.

All classifiers can now be compared on the same added-history populations, but their parameter spaces and search policies remain different. These exploratory studies do not establish general algorithm superiority. Keep the actual deployed checkpoint separate until a reviewed integration decision with matching serving/explanation exports. The anomaly-normalisation experiment is separate: it can be reported as sensitivity evidence now and considered for deployment only after validity review.